# Active Learning on Carboxylic Acid DFT Descriptors

**AI4Chemical Sciences Bootcamp, Caltech · 90 minutes · follows Lecture 13 (Active Learning)**

You have **8,528 commercially available carboxylic acids**. For each one there exist
**156 numbers**: 39 conformer-ensemble **DFT** descriptors × 4 aggregations over the
conformer ensemble (`_min`, `_max`, `_low_e` = lowest-energy conformer, `_boltz` =
Boltzmann average at 298 K).

The descriptors are hidden. You may **buy them for at most 600 molecules**, 50 at a
time. Your job is to train a Chemprop D-MPNN that predicts all 156 for a **hidden test
set you never see**, and to know how uncertain it is.

## The oracle is real

These labels are not a cheap surrogate. Every one was computed by

> Haas, B. C.; Hardy, M. A.; Sowndarya S. V., S.; Adams, K.; Coley, C. W.; Paton, R. S.;
> Sigman, M. S. *Digital Discovery* **2025**, *4*, 222–233.
> DOI [10.1039/D4DD00284A](https://doi.org/10.1039/D4DD00284A) · CC BY 4.0

at M06-2X/def2-TZVP // B3LYP-D3(BJ)/6-31G(d,p), over Maestro conformer ensembles with
GoodVibes quasi-harmonic Gibbs energies. Their paper reports **over 1,000,000 CPU
hours** for the acid library. The setup cell prints what that means per label.

## How this notebook is organised

| section | what you decide | ~time |
|---|---|---|
| **(a) Look at the data** | nothing yet — but form opinions | 15 min |
| **(b) Choose a GNN** | architecture, and critically the *predictor head* | 15 min |
| **(c) Choose an initialisation** | how to pick the first 100 molecules | 10 min |
| **(d) Choose an acquisition function** | the whole point of active learning | 30 min |
| **(e) Package and submit** | — | 10 min |

**Sections (a)–(c) cost you nothing.** They run on a free, fully-labelled `dev` set. That
is deliberate, and it is the lesson: rehearse on free data *before* you start queuing DFT
jobs. Only section (d) spends budget.

Run the cells in order. Everything stays in memory, so section (e) can package whatever
section (d) produced — including an acquisition function you wrote yourself.

## Setup

Run this once. On Colab it installs Chemprop (~2 min) and downloads the data bundle
(~10 MB) from the course repo. If the next cell fails with an import error, do
**Runtime → Restart session** and re-run this one.

Running locally instead? Clone the repo and start Jupyter from inside
`tutorials/13-dft-active-learning/`; the download step is skipped automatically.

In [ ]:
# --- setup: run me first -------------------------------------------------

# On Colab this installs Chemprop (~2 min) and downloads the tutorial files.

# If the next cell fails with an import error: Runtime > Restart session, re-run this.

import os, sys, subprocess, pathlib, urllib.request

# --- NumPy 2.0 compatibility shim -- must run BEFORE importing chemprop ----
# NumPy 2.0 moved these warning classes into np.exceptions. Some packages in the
# Chemprop dependency chain still read the old top-level names, which raises
#     AttributeError: module 'numpy' has no attribute 'VisibleDeprecationWarning'
# Restoring the aliases is additive and safe: they point at the very same classes.
import numpy as _np
for _name in ("VisibleDeprecationWarning", "ComplexWarning",
              "ModuleDeprecationWarning", "TooHardError", "AxisError"):
    if not hasattr(_np, _name) and hasattr(
            getattr(_np, "exceptions", None), _name):
        setattr(_np, _name, getattr(_np.exceptions, _name))
del _np, _name




IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules



REPO = "julschleinitz/ai4chemistry-bootcamp"

BRANCH = "main"

TUT = "tutorials/13-dft-active-learning"

RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/{TUT}"



# code + data the notebook needs. data/student/ is ~10 MB.

# Files marked optional degrade gracefully -- the notebook still runs without them.

OPTIONAL = {

    "data/student/published_benchmark.csv",   # only used by the comparison cell in (e)

    "data/student/README.md",

}

FILES = [

    "al_toolkit.py",

    "predict.py",

    "predict.py.sha256",

    "validate_submission.py",

    "leaderboard/submit_payload.py",

    "data/student/pool_meta.csv",

    "data/student/dev.csv",

    "data/student/selftest_smiles.csv",

    "data/student/pool_labels.enc",

    "data/student/targets.json",

    "data/student/published_benchmark.csv",

    "data/student/README.md",

]



if IN_COLAB:

    try:

        import chemprop  # noqa: F401

    except ImportError:

        subprocess.run([sys.executable, "-m", "pip", "install", "-q",

                        "chemprop==2.3.1"], check=True)

        print("chemprop installed -- if the next cell fails, "

              "Runtime > Restart session, then re-run this cell.\n")



    for rel in FILES:

        dest = pathlib.Path(rel)

        if dest.exists() and dest.stat().st_size > 0:

            continue

        dest.parent.mkdir(parents=True, exist_ok=True)

        try:

            urllib.request.urlretrieve(f"{RAW}/{rel}", dest)

        except Exception as exc:

            if rel in OPTIONAL:

                print(f"  (optional file {rel} unavailable -- continuing)")

                continue

            raise SystemExit(

                f"could not download {rel} ({type(exc).__name__}).\n"

                f"Check {RAW}/{rel} in a browser. If the repo is private or the "

                f"bundle has not been committed yet, ask the instructor."

            ) from exc

    have = sum(1 for r in FILES if pathlib.Path(r).exists())

    print(f"fetched {have}/{len(FILES)} files from {REPO}@{BRANCH}")



TUTORIAL_DIR = pathlib.Path.cwd()

BUNDLE = TUTORIAL_DIR / "data" / "student"

sys.path.insert(0, str(TUTORIAL_DIR))

sys.path.insert(0, str(TUTORIAL_DIR / "leaderboard"))



import json, dataclasses

import numpy as np, pandas as pd, matplotlib.pyplot as plt

import al_toolkit as al



plt.rcParams.update({"figure.dpi": 110, "font.size": 9,

                     "axes.spines.top": False, "axes.spines.right": False})



bundle = al.load_bundle(BUNDLE)

pool = bundle.pool_meta

dev = bundle.dev.reset_index(drop=True)



print(f"\npool  {len(pool):>6,} acids (labels hidden)")

print(f"dev   {len(dev):>6,} acids (labels free)")

print(f"targets {bundle.n_targets}  (+{len(bundle.extras)} unscored extras)")

print()

print(bundle.describe_oracle())

print()

print(f"one batch of 50 labels = {bundle.cpu_hours(50):>10,.0f} CPU-hours "

      f"= {bundle.cpu_hours(50) / 24:>6,.0f} CPU-days")

---

# (a) Look at the data

Before you model anything, look at what you are modelling. Three questions:

1. What chemistry is in the pool, and is it the chemistry *you* care about?
2. Which of the 156 targets are conformational, and which are not?
3. How far is the hidden test set from the pool?

## What is in the pool?

In [ ]:
print(pool[["mw", "n_heavy", "n_rot"]].describe().round(2).to_string())
print()
print(pool["subclass"].value_counts().to_string())
print(f"\nunique Murcko scaffolds in the pool: {pool['murcko_scaffold'].nunique():,}")
print(f"most common scaffold covers "
      f"{pool['murcko_scaffold'].value_counts().iloc[0]} molecules")

fig, axes = plt.subplots(1, 3, figsize=(9, 2.4))
for ax, col, label in zip(axes, ["mw", "n_heavy", "n_rot"],
                          ["MW / Da", "heavy atoms", "rotatable bonds"]):
    ax.hist(pool[col], bins=30, color="#4C72B0")
    ax.set_xlabel(label)
axes[0].set_ylabel("acids")
fig.tight_layout()

### A dozen of them

Ask yourself whether a model that only sees the 2D graph could tell which of these has a
sterically buried acidic proton.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

sample = pool.sample(12, random_state=1)
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in sample["smiles"]],
                     molsPerRow=4, subImgSize=(220, 160),
                     legends=[f"{r.acid_id}  {r.subclass}"
                              for r in sample.itertuples()])

## What the 39 descriptors are

Atom labels follow the paper, for the (R)₃C4–C1O2O3H5 group: **C1** carboxyl carbon,
**O2** carbonyl oxygen, **O3** hydroxyl oxygen, **H5** acidic hydrogen, **C4** α-carbon.

In [ ]:
spec = bundle.spec
fam_of = spec["family_of_target"]

desc = pd.DataFrame([
    {"base": b,
     "family": fam_of[f"{b}_boltz"],
     "unit": spec["unit_of_base"].get(b, "?"),
     "description": spec["description_of_base"].get(b, "")}
    for b in spec["base_properties"]])

print(desc.groupby("family").size().sort_values(ascending=False).to_string())
desc

**Notice the units — you will need this in section (d).** `%Vbur` runs 0–100,
`HOMO` is about −0.3 Hartree, `NMR_shift_C1` ~170 ppm, `IR_freq_C1_O2` ~1800 cm⁻¹,
`volume` is in the thousands of Bohr³, and `NBO_charge_H5` is about 0.5 e. Five orders of
magnitude across the target set.

## Which targets are conformational?

The authors computed, for every descriptor, a **Boltzmann-weighted standard deviation**
across the conformer ensemble. Those `_boltz_stdev` columns ship with your free dev set
and are *not* scored — they are here so you can ask:

> how much of this descriptor's variation is **conformational** rather than
> **structural**?

Divide the conformational spread by the structural spread (the std of `_boltz` across
molecules). Large means the descriptor moves more between conformers of one molecule than
between different molecules — and a 2D graph model never sees a conformer.

In [ ]:
base = spec["base_properties"]
rows = []
for b in base:
    bo, lo, hi = f"{b}_boltz", f"{b}_min", f"{b}_max"
    sd_col = f"{b}_boltz_stdev"
    structural = dev[bo].std()
    if not structural or not np.isfinite(structural) or structural <= 0:
        continue
    r = {"base": b, "family": fam_of[bo],
         "range_ratio": float(((dev[hi] - dev[lo]) / structural).mean())}
    if sd_col in dev.columns:
        r["stdev_ratio"] = float((dev[sd_col] / structural).mean())
    rows.append(r)
spread = pd.DataFrame(rows).sort_values("range_ratio")

colors = {"vbur": "#4C72B0", "charge": "#DD8452", "fmo": "#55A868",
          "spectroscopic": "#C44E52", "sterimol": "#8172B3",
          "sasa": "#937860", "geometry": "#DA8BC3", "electrostatic": "#8C8C8C"}
fig, ax = plt.subplots(figsize=(6.5, 7.5))
ax.barh(spread["base"], spread["range_ratio"],
        color=[colors[f] for f in spread["family"]])
ax.set_xlabel(r"mean $(\mathrm{max}-\mathrm{min})\ /\ \mathrm{std}_{\mathrm{molecules}}(\mathrm{boltz})$")
ax.set_title("how conformational is each descriptor?", loc="left")
ax.tick_params(axis="y", labelsize=6.5)
ax.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()],
          colors.keys(), frameon=False, fontsize=7, loc="lower right")
fig.tight_layout()

print("most conformational:")
print(spread.tail(5)[["base", "range_ratio"]].to_string(index=False))
print("\nleast conformational:")
print(spread.head(5)[["base", "range_ratio"]].to_string(index=False))

**Slow down here.**

The dihedrals sit at the top: `dihedral_O2_C1_O3_H5` is the syn/anti flip of the acid
proton, so its `_min` and `_max` are barely the same physical quantity. The NBO charges
and frontier-orbital terms sit at the bottom — they are close to properties of the graph,
and a GNN should do well on them.

So **the 156 targets are not equally hard, and not equally informative.** Remember that
when you write an acquisition function that sums uncertainty over all 156.

### How correlated are the targets?

In [ ]:
boltz = [f"{b}_boltz" for b in base]
Cm = dev[boltz].corr().to_numpy()
labels = [bundle.ascii(c).replace("_boltz", "") for c in boltz]

fig, ax = plt.subplots(figsize=(6.4, 5.6))
im = ax.imshow(Cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=5)
ax.set_yticklabels(labels, fontsize=5)
fig.colorbar(im, label="Pearson r", shrink=0.8)
ax.set_title("39 Boltzmann-averaged DFT descriptors", loc="left")
fig.tight_layout()

off = Cm[np.triu_indices_from(Cm, k=1)]
print(f"median |r| between descriptors: {np.nanmedian(np.abs(off)):.3f}")
print(f"fraction of pairs with |r| > 0.8: {np.nanmean(np.abs(off) > 0.8):.3f}")

The buried-volume block is visibly one thing measured three times, and the SASA /
polarisability / volume terms all track molecular size. So "156 targets" is optimistic —
the effective number of independent things to learn is much smaller. Good news for your
budget, and a hint about where an acquisition function should look.

### Optional: a real DFT conformer ensemble

The MolSSI API also serves the optimised geometries. Needs internet; nothing downstream
depends on it.

In [ ]:
import urllib.request

API = "https://descriptor-libraries.molssi.org/api/acids"

def conformer_xyz(acid_id, max_confs=6):
    with urllib.request.urlopen(f"{API}/molecules/{acid_id}", timeout=60) as r:
        rec = json.load(r)
    out = []
    for cid in rec.get("conformers_id", [])[:max_confs]:
        with urllib.request.urlopen(f"{API}/conformers/export/xyz/{cid}",
                                    timeout=60) as r:
            out.append((cid, r.read().decode()))
    return rec, out

try:
    aid = pool["acid_id"].iloc[0]
    rec, confs = conformer_xyz(aid)
    print(f"{aid}  {rec.get('smiles')}  -- "
          f"{len(rec.get('conformers_id', []))} conformers in the published ensemble")
    print(f"\nfirst conformer ({confs[0][0]}), first 6 lines:")
    print("\n".join(confs[0][1].splitlines()[:6]))
except Exception as exc:
    print(f"API unavailable ({type(exc).__name__}) -- skip this cell")

## Where is the hidden test set?

**Scaffold-disjoint.** Every Bemis–Murcko scaffold in it is absent from the pool and from
dev. You cannot buy it, or buy its neighbours. Below, a PCA of Morgan space with dev
overlaid on the pool — dev is scaffold-disjoint too, so the gap you see is roughly the gap
you are scored across.

In [ ]:
from sklearn.decomposition import PCA

sub_pool = pool.sample(min(2500, len(pool)), random_state=0)
X = al.morgan_matrix(list(sub_pool["smiles"]) + list(dev["smiles"])).astype(np.float32)
Z = PCA(n_components=2, random_state=0).fit_transform(X)
n1 = len(sub_pool)

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.scatter(Z[:n1, 0], Z[:n1, 1], s=4, alpha=.35, label="pool", color="#4C72B0")
ax.scatter(Z[n1:, 0], Z[n1:, 1], s=4, alpha=.55,
           label="dev (disjoint scaffolds)", color="#DD8452")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## Your turn

Answer these now. Section (d) will ask you to act on them.

In [ ]:
answers = {
    # look at the conformational-spread chart and the correlation map
    "hardest_targets": ["...", "...", "..."],
    # if you had to give up on one family, which?
    "family_i_would_sacrifice": "...",
    # from the subclass counts: what is thin in the pool?
    "under_represented_chemistry": "...",
}
answers

---

# (b) Choose a GNN

Chemprop is a directed message-passing neural network (D-MPNN): messages live on
*directed bonds* rather than atoms.

> Heid, E. *et al.* *J. Chem. Inf. Model.* **2024**, *64*, 9–17.
> DOI [10.1021/acs.jcim.3c01250](https://doi.org/10.1021/acs.jcim.3c01250)

Everything in this section runs on the **free dev set**. You spend no budget deciding what
model to use.

The most consequential choice is not the depth or the width. It is **the predictor head**,
because the head decides what uncertainty you can compute at all — and an
uncertainty-based acquisition function is only as good as its uncertainty.

## The practice split

700 dev molecules to train on, the rest to test. Sections (b) and (c) both use this.

In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(dev))
tr, te = perm[:700], perm[700:]

Xtr = dev.loc[tr, "smiles"].tolist()
Ytr = dev.loc[tr, bundle.targets].to_numpy(float)
Xte = dev.loc[te, "smiles"].tolist()
Yte = dev.loc[te, bundle.targets].to_numpy(float)
scales = al.target_scales(Yte)
print(f"practice: train {len(Xtr)}, test {len(Xte)}, targets {Ytr.shape[1]}")

## A baseline

`ModelSpec()` defaults are deliberately unremarkable. Fit it, see where you start.

In [ ]:
spec0 = al.ModelSpec(max_epochs=30, seed=0)
print(spec0.describe())

model0 = al.train_model(Xtr, Ytr, spec0, targets=bundle.targets)
mu, sg = model0.predict(Xte)
print(f"\nscaled MAE {al.scaled_mae(Yte, mu, scales):.4f}")
print(f"ENCE       {al.ence(Yte, mu, sg):.4f}   (nan means: no usable sigma)")

### Predicted vs true, across four families

In [ ]:
show = [t for t in ["NBO_charge_H5_boltz",
                    "Sterimol_L_C1_C4(Å)_morfeus_boltz",
                    "%Vbur_C1_3.0Å_boltz",
                    "dihedral_O2_C1_O3_H5(°)_max"] if t in bundle.targets]
fig, axes = plt.subplots(1, len(show), figsize=(2.7 * len(show), 2.9))
for ax, t in zip(np.atleast_1d(axes), show):
    j = bundle.targets.index(t)
    ax.scatter(Yte[:, j], mu[:, j], s=5, alpha=.4, color="#4C72B0")
    lim = [np.nanmin(Yte[:, j]), np.nanmax(Yte[:, j])]
    ax.plot(lim, lim, "k--", lw=.8)
    r = al.scaled_mae_per_task(Yte[:, [j]], mu[:, [j]], scales[[j]])[0]
    ax.set_title(f"{bundle.ascii(t)}\nsMAE {r:.2f}", fontsize=6.5)
    ax.set_xlabel("true"); ax.set_ylabel("predicted")
fig.tight_layout()

## Architecture sweep

Each row is one fit on 700 molecules. Don't over-read differences smaller than
seed-to-seed noise — which you can measure by rerunning one row with a different `seed`.

In [ ]:
trials = [
    dict(name="baseline"),
    dict(name="depth=2", depth=2),
    dict(name="depth=5", depth=5),
    dict(name="d_h=150", d_h=150),
    dict(name="d_h=600", d_h=600),
    dict(name="ffn_layers=2", ffn_layers=2),
    dict(name="dropout=0.1", dropout=0.1),
    dict(name="agg=norm", aggregation="norm"),
    dict(name="batch_norm", batch_norm=True),
]
rows = []
for t in trials:
    name = t.pop("name")
    s = al.ModelSpec(max_epochs=30, seed=0, **t)
    m = al.train_model(Xtr, Ytr, s, targets=bundle.targets)
    p, _ = m.predict(Xte)
    rows.append({"variant": name, "sMAE": al.scaled_mae(Yte, p, scales)})
    print(f"{name:<16} {rows[-1]['sMAE']:.4f}", flush=True)
pd.DataFrame(rows).sort_values("sMAE")

## The important choice: the head

| head | uncertainty you get | cost |
|---|---|---|
| `regression`, 1 model | **none** — `sigma` is all zeros | 1× |
| `mve`, 1 model | **aleatoric only** | 1× |
| `evidential`, 1 model | epistemic + aleatoric | 1× |
| `regression`, ensemble of 5 | epistemic (disagreement) | 5× |
| `mve`, ensemble of 3 | **epistemic + aleatoric** | 3× |

Lecture 13 §5 made the operational point: **uncertainty sampling on total uncertainty
keeps buying molecules whose labels are irreducibly noisy.** What you want to acquire on is
the *epistemic* part — the part more data actually fixes.

A single MVE model reports only aleatoric uncertainty, so it is precisely the wrong tool,
even though it is the cheapest thing that produces a `sigma`.

A wrinkle specific to this dataset: DFT labels are deterministic, so there is no
measurement noise. The "aleatoric" term instead absorbs **conformer-ensemble sampling
noise** — which the paper itself identifies as its dominant error source, especially for
the `_low_e` targets. That is genuinely irreducible at your budget, so the distinction
still bites.

In [ ]:
heads = [
    ("regression x1", dict(head="regression", ensemble_size=1)),
    ("mve x1",        dict(head="mve", ensemble_size=1)),
    ("evidential x1", dict(head="evidential", ensemble_size=1)),
    ("regression x4", dict(head="regression", ensemble_size=4)),
    ("mve x3",        dict(head="mve", ensemble_size=3)),
]
rows, fitted = [], {}
for name, kw in heads:
    s = al.ModelSpec(max_epochs=30, seed=0, **kw)
    m = al.train_model(Xtr, Ytr, s, targets=bundle.targets)
    p, sgm = m.predict(Xte)
    _, epi, ale = m.predict_components(Xte)
    fitted[name] = m
    rows.append({"head": name,
                 "sMAE": al.scaled_mae(Yte, p, scales),
                 "ENCE": al.ence(Yte, p, sgm),
                 "rho(sigma,|err|)": al.sigma_error_spearman(Yte, p, sgm),
                 "mean epistemic": float(np.nanmean(epi)),
                 "mean aleatoric": float(np.nanmean(ale)),
                 "cost (fits)": s.ensemble_size})
    print(f"{name:<15} sMAE {rows[-1]['sMAE']:.4f}  "
          f"ENCE {rows[-1]['ENCE']:.4f}  "
          f"rho {rows[-1]['rho(sigma,|err|)']:.3f}", flush=True)
pd.DataFrame(rows).set_index("head").round(4)

**What to notice.**

- `regression x1` has no `sigma` at all: ENCE is `nan`, and every uncertainty-based
  acquisition function silently degenerates to random.
- `mve x1` reports a `sigma`, but its epistemic column is zero. It cannot tell "I have
  never seen anything like this" from "this molecule's conformer ensemble is poorly
  sampled".
- The ensembles cost 3–5× the training time and are the only options giving a real
  epistemic signal.

You are scored on `mean_sMAE + mean_ENCE`. The second term is half the score and is
unobtainable without one of the bottom three rows.

### Epistemic vs aleatoric, per molecule

In [ ]:
m = fitted["mve x3"]
_, epi, ale = m.predict_components(Xte)
p, _ = m.predict(Xte)
err = np.abs(Yte - p) / scales

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3))
for ax, u, lab in ((axes[0], epi / scales, "epistemic"),
                   (axes[1], ale / scales, "aleatoric")):
    x = np.nanmean(u, axis=1); y = np.nanmean(err, axis=1)
    ax.scatter(x, y, s=6, alpha=.45, color="#4C72B0")
    ok = np.isfinite(x) & np.isfinite(y)
    ax.set_title(f"{lab}   Spearman {al._spearman(x[ok], y[ok]):.2f}", fontsize=9)
    ax.set_xlabel(f"mean scaled {lab} sigma")
    ax.set_ylabel("mean scaled |error|")
fig.tight_layout()

## Commit to a model

Two practical constraints:

- Each fit happens **11 times** in a 10-round loop. `ensemble_size=5` with
  `max_epochs=100` will not finish in the session. Budget roughly
  `ensemble_size × max_epochs × 1 s`.
- You need a non-degenerate `sigma`. If your ENCE above was `nan`, go back.

In [ ]:
my_spec = al.ModelSpec(
    head="mve",            # regression | mve | evidential
    ensemble_size=3,       # >1 for epistemic uncertainty
    depth=3,
    d_h=300,
    dropout=0.0,
    aggregation="mean",
    max_epochs=40,
    batch_size=32,
    seed=0,
)
print(my_spec.describe())

# saved as a record of your choice, and as insurance if the runtime dies
json.dump(dataclasses.asdict(my_spec), open("my_spec.json", "w"), indent=2)
print("\nsaved my_spec.json")

---

# (c) Choose an initialisation

Round 0 has no model, so it cannot have an acquisition function. You pick 100 molecules
blind. This is the **cold-start problem**, and it is where the largest and most reliable
gains in this exercise live — every later round inherits whatever your seed taught the
model.

Four methods, same budget:

| method | idea |
|---|---|
| `random` | the honest baseline |
| `maxmin` | MaxMin diversity picker on Morgan fingerprints |
| `kmeans` | 100 clusters in Morgan space, take the molecule nearest each centroid |
| `scaffold_balanced` | round-robin across Bemis–Murcko scaffolds |

Still no budget spent: we use dev as a stand-in pool, which is how you would rehearse a
real campaign.

In [ ]:
practice_spec = al.ModelSpec(**{**dataclasses.asdict(my_spec),
                                "max_epochs": min(my_spec.max_epochs, 30)})

practice_pool = dev.loc[perm[:800]].reset_index(drop=True)
practice_test = dev.loc[perm[800:]].reset_index(drop=True)
Yp = practice_test[bundle.targets].to_numpy(float)
scales_p = al.target_scales(Yp)
print(f"practice pool {len(practice_pool)}, practice test {len(practice_test)}")

## What does each seed set look like?

Before training anything, measure the seed sets themselves:

- **mean pairwise Tanimoto** inside the seed — lower is more diverse
- **distinct scaffolds** — higher is broader
- **coverage distance**: mean distance from *every* pool molecule to its nearest seed
  molecule. This is the one that matters, and it is what core-set theory is about — low
  coverage means "wherever the test molecule lands, it has a neighbour in my training
  set".

In [ ]:
from rdkit import DataStructs

N_SEED = 100
fps_pool = al.rdkit_fps(practice_pool["smiles"].tolist())

def coverage(seed_ids):
    idx = [practice_pool.index[practice_pool.acid_id == a][0] for a in seed_ids]
    seed_fps = [fps_pool[i] for i in idx]
    return float(np.mean([1.0 - max(DataStructs.BulkTanimotoSimilarity(f, seed_fps))
                          for f in fps_pool]))

seeds, rows = {}, []
for name, fn in al.SEED_SELECTORS.items():
    ids = fn(practice_pool, N_SEED, seed=0)
    seeds[name] = ids
    sub = practice_pool[practice_pool.acid_id.isin(ids)]
    rows.append({"method": name,
                 "mean pairwise Tanimoto":
                     al.mean_pairwise_tanimoto(sub["smiles"].tolist()),
                 "distinct scaffolds": sub["murcko_scaffold"].nunique(),
                 "coverage distance": coverage(ids),
                 "mean MW": sub["mw"].mean()})
    print(f"{name:<20} done", flush=True)
seed_stats = pd.DataFrame(rows).set_index("method")

fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.7))
for ax, col, better in zip(axes,
        ["mean pairwise Tanimoto", "distinct scaffolds", "coverage distance"],
        ["lower = more diverse", "higher = broader", "lower = better covered"]):
    ax.bar(seed_stats.index, seed_stats[col], color="#4C72B0")
    ax.set_title(f"{col}\n({better})", fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
fig.tight_layout()
seed_stats.round(3)

## Does it show up in the model?

Same spec, same 100 molecules of budget, four seed sets.

In [ ]:
seed_results = {}
for name, ids in seeds.items():
    sub = practice_pool[practice_pool.acid_id.isin(ids)]
    m = al.train_model(sub["smiles"].tolist(),
                       sub[bundle.targets].to_numpy(float),
                       practice_spec, targets=bundle.targets)
    p, sgm = m.predict(practice_test["smiles"].tolist())
    seed_results[name] = {"sMAE": al.scaled_mae(Yp, p, scales_p),
                          "ENCE": al.ence(Yp, p, sgm)}
    print(f"{name:<20} sMAE {seed_results[name]['sMAE']:.4f}  "
          f"ENCE {seed_results[name]['ENCE']:.4f}", flush=True)

res = pd.DataFrame(seed_results).T.sort_values("sMAE")
fig, ax = plt.subplots(figsize=(4.6, 2.8))
ax.bar(res.index, res["sMAE"], color="#4C72B0")
ax.set_ylabel("scaled MAE after 100 labels")
ax.tick_params(axis="x", rotation=30, labelsize=8)
ax.set_ylim(0, max(res["sMAE"]) * 1.15)
fig.tight_layout()
res.round(4)

## Seed variance — the number that keeps you honest

Run `random` five times with different seeds. If the spread of those five is as large as
the gap between your best and worst method above, the comparison you just did measured
nothing.

This is the check Lecture 13 §8 asked for, and the reason "AL beat random" claims so often
fail to replicate.

In [ ]:
vals = []
for s in range(5):
    ids = al.seed_random(practice_pool, N_SEED, seed=s)
    sub = practice_pool[practice_pool.acid_id.isin(ids)]
    m = al.train_model(sub["smiles"].tolist(),
                       sub[bundle.targets].to_numpy(float),
                       al.ModelSpec(**{**dataclasses.asdict(practice_spec),
                                       "seed": s}),
                       targets=bundle.targets)
    p, _ = m.predict(practice_test["smiles"].tolist())
    vals.append(al.scaled_mae(Yp, p, scales_p))
    print(f"  random seed {s}: {vals[-1]:.4f}", flush=True)

vals = np.array(vals)
gap = res["sMAE"].max() - res["sMAE"].min()
print(f"\nrandom baseline: {vals.mean():.4f} +/- {vals.std(ddof=1):.4f} "
      f"(range {np.ptp(vals):.4f})")
print(f"best-worst gap between seed methods: {gap:.4f}")
print("\n" + ("The method differences are LARGER than seed noise -- believe them."
              if gap > 2 * vals.std(ddof=1) else
              "The method differences are WITHIN seed noise -- do not believe them yet."))

## Commit to a seed method

If the answer is "random", say so and mean it. That is a legitimate finding, and more
useful to the room than a method chosen because it sounded sophisticated.

In [ ]:
MY_SEED_METHOD = "maxmin"    # random | maxmin | kmeans | scaffold_balanced
MY_REASON = "..."

json.dump({"seed_method": MY_SEED_METHOD, "reason": MY_REASON},
          open("my_seed.json", "w"), indent=2)
print(f"chose {MY_SEED_METHOD}: {MY_REASON}")

---

# (d) Choose an acquisition function

**This is the section that matters.**

```
seed 100  ->  fit  ->  score every unlabelled candidate  ->  buy the top 50
          ->  fit  ->  score  ->  buy 50  ->  ...  (10 rounds)  ->  600 labels
```

Everything except the third step is fixed. The acquisition function is the entire
intellectual content of active learning.

**600 labels per campaign.** You may run several campaigns — each gets its own `Oracle`
and its own log — and submit the best. But in a real lab you get one, which is exactly why
(b) and (c) happened on free data.

**Run `random` first.** Not as a formality: as the thing you have to beat.

In [ ]:
print(my_spec.describe())
print(f"\nseed method: {MY_SEED_METHOD}")
print(f"budget: 600 labels ~ {bundle.cpu_hours(600):,.0f} CPU-hours of DFT")
print("\navailable acquisition functions:")
for k in al.ACQUISITION:
    print("  ", k)

runs = {}   # every campaign you run lands here

## 1. The mandatory baseline

~8 minutes with an ensemble of 3. Start it, then read the next cell while it runs.

In [ ]:
runs["random"] = al.run_al_loop(
    bundle, my_spec, acquisition="random", seed_method=MY_SEED_METHOD,
    log_path="al_log_random.jsonl", seed=0)

## 2. The obvious thing, and why it fails

`max_variance` takes the top 50 by **raw** summed predictive standard deviation across all
156 targets.

Look back at the units you catalogued in (a):

| descriptor | typical magnitude |
|---|---|
| `volume(Bohr_radius³/mol)` | ~1,000–3,000 |
| `IR_freq_C1_O2` | ~1,800 cm⁻¹ |
| `NMR_shift_C1` | ~170 ppm |
| `%Vbur_C1_3.0Å` | ~60 |
| `NBO_charge_H5` | ~0.5 e |
| `HOMO` | ~−0.3 Hartree |

A raw sum of standard deviations over those is, to three significant figures, a sum over
`volume` and `IR_freq`. The acquisition function will spend your entire 600-label budget
improving two descriptors out of thirty-nine — and you will never find out from the score,
because the leaderboard metric scales each task.

Expect it to be **no better than random, plausibly worse.**

In [ ]:
runs["max_variance"] = al.run_al_loop(
    bundle, my_spec, acquisition="max_variance", seed_method=MY_SEED_METHOD,
    log_path="al_log_maxvar.jsonl", seed=0)

## 3. The one-line fix

In [ ]:
runs["max_variance_scaled"] = al.run_al_loop(
    bundle, my_spec, acquisition="max_variance_scaled",
    seed_method=MY_SEED_METHOD, log_path="al_log_maxvar_scaled.jsonl", seed=0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
al.plot_learning_curves(runs, "dev_scaled_mae", ax=axes[0])
al.plot_learning_curves(runs, "dev_ence", ax=axes[1])
al.plot_learning_curves(runs, "dev_spearman", ax=axes[2])
fig.tight_layout()

pd.DataFrame({k: {"final sMAE": v.to_frame()["dev_scaled_mae"].iloc[-1],
                  "final ENCE": v.to_frame()["dev_ence"].iloc[-1],
                  "AULC": al.aulc(v)}
              for k, v in runs.items()}).T.round(4)

### Where did `max_variance` spend the budget?

Per-family scaled MAE at the end of each campaign. This is the diagnosis, not just the
symptom.

In [ ]:
fams = bundle.spec["families"]
tab = pd.DataFrame({k: {f: v.to_frame()[f"smae_{f}"].iloc[-1] for f in fams
                        if f"smae_{f}" in v.to_frame().columns}
                    for k, v in runs.items()})
ax = tab.plot.bar(figsize=(8, 3), rot=20, width=.8)
ax.set_ylabel("final scaled MAE"); ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
tab.round(4)

## 4. Batch redundancy

Top-$k$ by uncertainty picks the 50 *individually* most uncertain molecules. If the model
is uncertain about one 3-fluorobenzoic acid, it is uncertain about all forty of them, and
you have just bought the same information forty times — at ~117 CPU-hours each.

> Kirsch, A.; van Amersfoort, J.; Gal, Y. **BatchBALD.** NeurIPS **2019**.
> arXiv [1906.08158](https://arxiv.org/abs/1906.08158)

`batch_diverse_topk` takes the top 200 by uncertainty then MaxMins down to 50 — the cheap
surrogate. `badge` (Ash *et al.*, ICLR 2020) does it more gracefully with k-means++ on
uncertainty-scaled embeddings.

The diagnostic is free: mean pairwise Tanimoto **inside each acquired batch**.

In [ ]:
for name in ["batch_diverse_topk", "badge"]:
    runs[name] = al.run_al_loop(
        bundle, my_spec, acquisition=name, seed_method=MY_SEED_METHOD,
        log_path=f"al_log_{name}.jsonl", seed=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
al.plot_learning_curves(runs, "dev_scaled_mae", ax=axes[0])
for k, v in runs.items():
    df = v.to_frame()
    if "batch_mean_tanimoto" in df.columns:
        axes[1].plot(df["n_labels"], df["batch_mean_tanimoto"], marker="o", label=k)
axes[1].set_xlabel("labels bought")
axes[1].set_ylabel("mean pairwise Tanimoto\nwithin the acquired batch")
axes[1].legend(frameon=False, fontsize=7)
fig.tight_layout()

## 5. Epistemic vs total, and BALD

`max_epistemic` uses only the ensemble-disagreement part of the variance.
`bald_ensemble` computes the Gaussian mutual information

$$ I \;=\; \tfrac{1}{2}\log \sigma^2_{\text{total}} \;-\; \tfrac{1}{2}\log \sigma^2_{\text{aleatoric}} $$

summed over tasks — epistemic **relative to** irreducible noise. On this dataset the
"irreducible" part is conformer-ensemble sampling noise, which the paper identifies as its
own dominant error source, so the distinction is not academic.

`uncertainty_times_novelty` is the closest thing here to the acquisition function in
Schleinitz *et al.*, *JACS* **2025**, *147*, 7476: what you do not know, times how far it
is from what you already have.

Pick one or two — you will not have time for all of them.

In [ ]:
for name in ["max_epistemic", "bald_ensemble", "uncertainty_times_novelty"][:2]:
    runs[name] = al.run_al_loop(
        bundle, my_spec, acquisition=name, seed_method=MY_SEED_METHOD,
        log_path=f"al_log_{name}.jsonl", seed=0)

fig, ax = plt.subplots(figsize=(6, 3.6))
al.plot_learning_curves(runs, "dev_scaled_mae", ax=ax)
fig.tight_layout()

## 6. Write your own

Define it here and pass the function straight to `run_al_loop`. (There is also a
`your_own` stub in `al_toolkit.py` if you prefer to edit the file.)

```python
def my_af(k: int, cand) -> np.ndarray:
    return positions   # k integer indices into cand.acid_ids
```

Available: `cand.mean`, `cand.sigma`, `cand.sigma_epistemic`, `cand.sigma_aleatoric` (all
`(n, 156)`), `cand.emb` and `cand.labeled_emb` (learned 300-d embeddings), `cand.smiles`,
`cand.scales`, and `cand.scaled(x)` to normalise per task. Use `cand.rng`, not
`np.random`, so your run reproduces.

Ideas worth an attempt:

- **Weight by how unusual the prediction is.** Buy molecules that are both uncertain and
  predicted to be extreme — the descriptor-space analogue of "buy the reactive sites" from
  the JACS paper.
- **Only count the families you care about.** You named a family to sacrifice in (a). Act
  on it.
- **Target the conformational descriptors.** The dihedrals moved most in (a); a molecule
  whose predicted `_max − _min` gap is large is one where the ensemble is doing real work,
  so one label buys more information.
- **Penalise within-batch similarity explicitly** rather than via MaxMin.

The example below is a starting point to replace, not a good answer.

In [ ]:
def my_af(k, cand):
    """Epistemic x extremeness, with a within-batch decorrelation pass."""
    u = np.nan_to_num(np.nansum(cand.scaled(cand.sigma_epistemic), axis=1))
    z = np.nan_to_num(np.nanmean(np.abs(cand.scaled(cand.mean)), axis=1))

    def norm(v):
        return (v - v.min()) / (np.ptp(v) + 1e-12)

    score = norm(u) * (0.5 + 0.5 * norm(z))
    order = np.argsort(-score)
    picked, emb = [], cand.emb
    for i in order:
        if len(picked) >= k:
            break
        if picked and np.linalg.norm(emb[picked] - emb[i], axis=1).min() < 1e-6:
            continue
        picked.append(int(i))
    while len(picked) < k:
        for i in order:
            if int(i) not in picked:
                picked.append(int(i)); break
    return np.array(picked[:k])


runs["my_af"] = al.run_al_loop(
    bundle, my_spec, acquisition=my_af, seed_method=MY_SEED_METHOD,
    log_path="al_log_mine.jsonl", seed=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
al.plot_learning_curves(runs, "dev_scaled_mae", ax=axes[0])
al.plot_learning_curves(runs, "dev_ence", ax=axes[1])
fig.tight_layout()

summary = pd.DataFrame({
    k: {"final sMAE": v.to_frame()["dev_scaled_mae"].iloc[-1],
        "final ENCE": v.to_frame()["dev_ence"].iloc[-1],
        "combined": (v.to_frame()["dev_scaled_mae"].iloc[-1]
                     + v.to_frame()["dev_ence"].iloc[-1]),
        "AULC": al.aulc(v),
        "labels": v.oracle.spent}
    for k, v in runs.items()}).T.sort_values("combined")
summary.round(4)

## 7. Choose your submission

`combined = sMAE + ENCE` on **dev** is your best available estimate of the leaderboard
score. It is an estimate: dev is scaffold-disjoint from the pool, but so is the test set,
and they are different draws.

Because this is all one notebook, `best_run` below is the *actual* model object you
selected — including a custom `my_af` campaign.

In [ ]:
BEST = summary.index[0]
best_run = runs[BEST]
print(f"best on dev: {BEST}  "
      f"(acquisition={best_run.config['acquisition']}, "
      f"{best_run.oracle.spent} labels)")
best_run.to_frame().round(4)

---

# (e) Package and submit

```
submission_<team>/
    manifest.json        your choices + the target order
    models/*.pt          your Chemprop checkpoint(s)
    predict.py           PROVIDED, UNMODIFIED (sha256-checked)
    al_log.jsonl         the oracle's append-only purchase log
    learning_curve.csv   dev metrics after every round
```

Then `validate_submission.py` runs the same checks the instructor's scorer will, **and
actually runs your `predict.py` on 50 public molecules**. Do not upload anything until it
prints `READY TO SUBMIT`.

You are scored on the hidden 1,000 acids, whose scaffolds appear nowhere in your pool:

$$\text{combined} \;=\; \underbrace{\tfrac{1}{156}\sum_t \frac{\mathrm{MAE}_t}{\sigma_t}}_{\text{accuracy}} \;+\; \underbrace{\tfrac{1}{156}\sum_t \mathrm{ENCE}_t}_{\text{calibration}}$$

Lower is better. You can package any campaign — set `best_run = runs["..."]` first if you
want a different one.

## 1. Package

In [ ]:
TEAM = "team_name_here"          # <-- change me
NOTES = "one sentence on what you did and why"

sub = al.package_submission(best_run, bundle, team=TEAM, notes=NOTES)

## 2. Self-test — the gate

Files, budget, log, target order, `predict.py` checksum, then end-to-end inference.

In [ ]:
proc = subprocess.run(
    [sys.executable, "validate_submission.py",
     "--submission-dir", str(sub), "--bundle", str(BUNDLE)],
    capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-2000:])
    raise SystemExit("validation failed -- fix the problems above, then re-run")

### Common failures and what they mean

| message | fix |
|---|---|
| `predict.py has been modified` | copy the pristine `predict.py` back in; your logic belongs in the acquisition function, not the predictor |
| `every sigma is zero` | you used `head="regression"` with `ensemble_size=1`; you cannot be calibrated without an uncertainty estimate |
| `targets are the right set but the WRONG ORDER` | use `bundle.targets` verbatim, never `sorted()`. The buried-volume names contain the substrings `min` and `max`, so sorting them is especially easy to get wrong |
| `budget EXCEEDED` | you packaged a run that bought more than 600 |
| `manifest labels_used disagrees with the log` | you mixed a manifest from one campaign with a log from another |

## 3. Post your dev score to the live leaderboard

**This board is not the result.** You are reporting your score on the *public dev set*, because
you cannot compute your score on the hidden test set — you have never seen those molecules.

After the session the instructor runs every submitted checkpoint against the scaffold-disjoint
hidden test set and publishes the real ranking. Teams move between the two boards, sometimes a
lot. Watching who moves is the point: a team that tuned hard against dev will fall, and that is
what "scaffold-disjoint generalisation" means in practice.

Submit as often as you like — the board keeps your best `dev_combined`. Keep `TEAM` identical
across runs so it can.

In [ ]:
# The instructor pastes the Apps Script Web App URL here before class.
LEADERBOARD_ENDPOINT_URL = "PASTE_WEB_APP_URL_HERE"

import submit_payload as lb

payload = lb.build_payload(best_run, bundle, team_name=TEAM, notes=NOTES)
lb.write_payload_json(payload, "leaderboard_submission.json")
lb.append_payload_csv(payload, "local_leaderboard_log.csv")

print(json.dumps(payload, indent=2))

if LEADERBOARD_ENDPOINT_URL.startswith("PASTE_"):
    print("\n[leaderboard endpoint not configured -- payload saved locally, "
          "nothing posted]")
else:
    try:
        print("\n", lb.submit_payload(payload, LEADERBOARD_ENDPOINT_URL))
        print("posted. Refresh the leaderboard on the tutorial page.")
    except Exception as exc:
        print(f"\npost failed ({type(exc).__name__}: {exc}) -- your run is still "
              "saved in leaderboard_submission.json; hand it to the instructor.")

## 4. Zip and upload

In [ ]:
import shutil
zip_path = shutil.make_archive(str(sub), "zip", root_dir=sub.parent,
                               base_dir=sub.name)
print(f"created {zip_path}  "
      f"({pathlib.Path(zip_path).stat().st_size / 1e6:.1f} MB)")

In [ ]:
DRIVE_FOLDER = ("/content/drive/MyDrive/AI4Chem Bootcamp 2026/"
                "lecture_13_tutorial/submissions")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dest = pathlib.Path(DRIVE_FOLDER)
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, dest)
    print(f"uploaded to {dest / pathlib.Path(zip_path).name}")
else:
    print(f"upload {zip_path} to the shared Drive folder:")
    print("  AI4Chem Bootcamp 2026 / lecture_13_tutorial / submissions/")

## 5. How do you compare to the published model?

The paper trained its own GNN on these exact descriptors with **7,290 training
molecules**. You used at most 600.

In [ ]:
bench = bundle.published_benchmark
if bench is None:
    print("published_benchmark.csv not in the bundle -- skip")
else:
    Y = dev[bundle.targets].to_numpy(float)
    mu_dev, _ = best_run.final_model.predict(dev["smiles"].tolist())
    rows = []
    for r in bench.itertuples(index=False):
        j = bundle.targets.index(r.target)
        mae = float(np.nanmean(np.abs(Y[:, j] - mu_dev[:, j])))
        rows.append({"target": bundle.ascii(r.target),
                     "your MAE (dev)": mae,
                     "published MAE": r.published_3D_GNN_MAE,
                     "ratio": mae / r.published_3D_GNN_MAE})
    cmp = pd.DataFrame(rows).sort_values("ratio")
    print(f"your training set:  {best_run.oracle.spent:>6,} molecules")
    print(f"their training set: {int(bench['published_train_size'].iloc[0]):>6,} molecules")
    print(f"\nmedian MAE ratio: {cmp['ratio'].median():.2f}x\n")
    display(cmp.round(3))

**Read the caveat before quoting any of this.** Their test set was a **random**
split; yours is scaffold-disjoint, which is strictly harder. Their numbers are a ceiling,
not a like-for-like target. A ratio of 2× on 600 molecules against 7,290 is a good result.

## 6. While the leaderboard runs — three questions

1. **Did your acquisition function beat random?** If not, say so out loud. A negative
   result honestly reported is the most useful thing in this literature; Lecture 13 §8
   exists because so few people report it.

2. **Where does your remaining error sit?** Check the per-family breakdown. A team that
   wins on NBO charges and loses on the dihedrals has learned something specific about
   what a 2D graph can and cannot encode.

3. **You spent 600 of 8,528 labels — 7%, about 70,000 CPU-hours.** Where would your
   learning curve have to flatten before you would responsibly tell a collaborator to stop
   the calculations? That question, not the leaderboard, is the point of the tutorial.